# OCRP Macro Conv Weight Probe

This notebook loads the trained OCRP macro checkpoint and inspects exactly what the LR and HR conv blocks learned.

Focus:

- which tensors in `conv_lr1` and `conv_hr1` are actually trainable
- the exact learned values from `best_model.pt`
- how the 8 tensor-product weights map onto the `2e/4e` coupling paths
- what the learned spatial priors look like for LR and HR conv

Important interpretation note:

- `spatial_logits` are learned spatial priors
- the runtime neighborhood weights are still modulated by the cosine-similarity mask and then renormalized
- so the softmax heatmaps below show the learned prior, not the final per-sample spatial weights

In [1]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "models").exists():
    repo_root = repo_root.parent
if not (repo_root / "models").exists():
    raise RuntimeError("Could not locate repo root containing a models/ directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.probe_ocrp_macro_stages import _load_ocrp_model_from_checkpoint

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)
pd.set_option("display.max_colwidth", None)

In [2]:
EXP_DIR = repo_root / "experiments/IN718/iso_embedding_ocrp_macro_01"
RUN_CONFIG = EXP_DIR / "logs/run_config.json"
CHECKPOINT = EXP_DIR / "checkpoints/best_model.pt"

assert RUN_CONFIG.exists(), f"Missing run config: {RUN_CONFIG}"
assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"

cfg = SimpleNamespace(**json.loads(RUN_CONFIG.read_text()))
model = _load_ocrp_model_from_checkpoint(cfg, CHECKPOINT, device=torch.device("cpu"))

print("EXP_DIR    :", EXP_DIR)
print("RUN_CONFIG :", RUN_CONFIG)
print("CHECKPOINT :", CHECKPOINT)
print("MODEL      :", type(model).__name__)
print("feature_irreps cfg:", cfg.feature_irreps)
print("model.irreps_feat :", model.irreps_feat)

EXP_DIR    : /data/home/umang/Materials/Reynolds-QSR_4x1/experiments/IN718/iso_embedding_ocrp_macro_01
RUN_CONFIG : /data/home/umang/Materials/Reynolds-QSR_4x1/experiments/IN718/iso_embedding_ocrp_macro_01/logs/run_config.json
CHECKPOINT : /data/home/umang/Materials/Reynolds-QSR_4x1/experiments/IN718/iso_embedding_ocrp_macro_01/checkpoints/best_model.pt
MODEL      : IsoEmbeddingSROCRP
feature_irreps cfg: full
model.irreps_feat : 1x2e+1x4e


## What the conv blocks are in this checkpoint

For this OCRP macro run, both convs operate in the same feature space:

- `irreps_in = irreps_out = 1x2e + 1x4e`
- each TP has 8 learned coupling-path weights
- because `irreps_in == irreps_out`, there is no learned residual projection module
- the trainable conv tensors are only `spatial_logits` and `tp.weight`

In [3]:
def tensor_rows(named_tensors, *, kind):
    rows = []
    for name, tensor in named_tensors:
        rows.append(
            {
                "kind": kind,
                "name": name,
                "shape": tuple(tensor.shape),
                "numel": int(tensor.numel()),
                "dtype": str(tensor.dtype),
                "requires_grad": bool(getattr(tensor, "requires_grad", False)),
            }
        )
    return rows


def conv_overview(name, conv):
    return {
        "conv": name,
        "kernel_size": int(conv.kernel_size),
        "irreps_in": str(conv.irreps_in),
        "irreps_out": str(conv.irreps_out),
        "use_residual": bool(conv.use_residual),
        "tp_weight_numel": int(conv.tp.weight_numel),
        "parameter_names": [n for n, _ in conv.named_parameters()],
        "buffer_names": [n for n, _ in conv.named_buffers()],
    }


overview_df = pd.DataFrame(
    [
        conv_overview("conv_lr1", model.conv_lr1),
        conv_overview("conv_hr1", model.conv_hr1),
    ]
)
display(overview_df)

for name in ["conv_lr1", "conv_hr1"]:
    conv = getattr(model, name)
    tensors_df = pd.DataFrame(
        tensor_rows(conv.named_parameters(), kind="parameter")
        + tensor_rows(conv.named_buffers(), kind="buffer")
    )
    print(f"\n{name}")
    display(tensors_df)

,conv,kernel_size,irreps_in,irreps_out,use_residual,tp_weight_numel,parameter_names,buffer_names
0,conv_lr1,5,1x2e+1x4e,1x2e+1x4e,True,8,"[spatial_logits, tp.weight]","[center_mask, tp.output_mask, tp._compiled_main_left_right._w3j_2_2_2, tp._compiled_main_left_right._w3j_2_2_4, tp._compiled_main_left_right._w3j_2_4_2, tp._compiled_main_left_right._w3j_2_4_4, tp._compiled_main_left_right._w3j_4_2_2, tp._compiled_main_left_right._w3j_4_2_4, tp._compiled_main_left_right._w3j_4_4_2, tp._compiled_main_left_right._w3j_4_4_4]"
1,conv_hr1,7,1x2e+1x4e,1x2e+1x4e,True,8,"[spatial_logits, tp.weight]","[center_mask, tp.output_mask, tp._compiled_main_left_right._w3j_2_2_2, tp._compiled_main_left_right._w3j_2_2_4, tp._compiled_main_left_right._w3j_2_4_2, tp._compiled_main_left_right._w3j_2_4_4, tp._compiled_main_left_right._w3j_4_2_2, tp._compiled_main_left_right._w3j_4_2_4, tp._compiled_main_left_right._w3j_4_4_2, tp._compiled_main_left_right._w3j_4_4_4]"



conv_lr1


,kind,name,shape,numel,dtype,requires_grad
0,parameter,spatial_logits,"(5, 5)",25,torch.float32,True
1,parameter,tp.weight,"(8,)",8,torch.float32,True
2,buffer,center_mask,"(1, 1, 1, 5, 5)",25,torch.bool,False
3,buffer,tp.output_mask,"(14,)",14,torch.float32,False
4,buffer,tp._compiled_main_left_right._w3j_2_2_2,"(5, 5, 5)",125,torch.float32,False
5,buffer,tp._compiled_main_left_right._w3j_2_2_4,"(5, 5, 9)",225,torch.float32,False
6,buffer,tp._compiled_main_left_right._w3j_2_4_2,"(5, 9, 5)",225,torch.float32,False
7,buffer,tp._compiled_main_left_right._w3j_2_4_4,"(5, 9, 9)",405,torch.float32,False
8,buffer,tp._compiled_main_left_right._w3j_4_2_2,"(9, 5, 5)",225,torch.float32,False
9,buffer,tp._compiled_main_left_right._w3j_4_2_4,"(9, 5, 9)",405,torch.float32,False



conv_hr1


,kind,name,shape,numel,dtype,requires_grad
0,parameter,spatial_logits,"(7, 7)",49,torch.float32,True
1,parameter,tp.weight,"(8,)",8,torch.float32,True
2,buffer,center_mask,"(1, 1, 1, 7, 7)",49,torch.bool,False
3,buffer,tp.output_mask,"(14,)",14,torch.float32,False
4,buffer,tp._compiled_main_left_right._w3j_2_2_2,"(5, 5, 5)",125,torch.float32,False
5,buffer,tp._compiled_main_left_right._w3j_2_2_4,"(5, 5, 9)",225,torch.float32,False
6,buffer,tp._compiled_main_left_right._w3j_2_4_2,"(5, 9, 5)",225,torch.float32,False
7,buffer,tp._compiled_main_left_right._w3j_2_4_4,"(5, 9, 9)",405,torch.float32,False
8,buffer,tp._compiled_main_left_right._w3j_4_2_2,"(9, 5, 5)",225,torch.float32,False
9,buffer,tp._compiled_main_left_right._w3j_4_2_4,"(9, 5, 9)",405,torch.float32,False


## Raw learned tensors from `best_model.pt`

This is the direct answer to “what weights are learned?” for the two convs.

In [4]:
for name in ["conv_lr1", "conv_hr1"]:
    conv = getattr(model, name)
    print(f"\n=== {name} ===")
    print("spatial_logits:")
    print(conv.spatial_logits.detach().cpu())
    print("\ntp.weight:")
    print(conv.tp.weight.detach().cpu())


=== conv_lr1 ===
spatial_logits:
tensor([[-0.1666, -0.1357, -0.3341, -0.0281,  0.1089],
        [-0.1108, -0.0300, -0.2640,  0.1010,  0.2693],
        [-0.2301, -0.1750, -1.3157, -0.1651,  0.1694],
        [ 0.0332, -0.0306, -0.0959,  0.2688,  0.6472],
        [ 0.1900,  0.2921,  0.3737,  0.6413,  0.9136]])

tp.weight:
tensor([ 0.3464,  0.6354,  0.2577, -0.8142, -1.4372, -0.3402,  2.0318,  0.9768])

=== conv_hr1 ===
spatial_logits:
tensor([[     0.0002,     -0.1370,     -0.0398,      0.5299,      0.0033,
              0.0305,      0.2631],
        [     0.0908,     -0.0459,      0.0084,      0.6867,     -0.0425,
              0.0251,      0.3518],
        [     0.4925,      0.3302,     -0.1048,     -0.2379,     -0.3841,
              0.0144,      0.5277],
        [     1.0325,      1.0100,     -0.1614,     -0.9339,     -0.5989,
              0.0885,      0.7997],
        [     0.4808,      0.1937,     -0.4659,     -0.7400,     -0.7273,
             -0.3745,      0.3583],
        [    

In [5]:
def spatial_prior(conv):
    logits = conv.spatial_logits.detach().cpu()
    return torch.softmax(logits.reshape(-1), dim=0).reshape_as(logits)


def top_kernel_rows(conv, k=6):
    prior = spatial_prior(conv)
    flat = prior.reshape(-1)
    values, indices = torch.topk(flat, k=min(k, flat.numel()))
    rows = []
    w = prior.shape[1]
    center_r = prior.shape[0] // 2
    center_c = prior.shape[1] // 2
    for rank, (value, idx) in enumerate(zip(values.tolist(), indices.tolist()), start=1):
        r = idx // w
        c = idx % w
        rows.append(
            {
                "rank": rank,
                "row": r,
                "col": c,
                "offset": (r - center_r, c - center_c),
                "prior_weight": value,
            }
        )
    center_value = float(prior[center_r, center_c].item())
    center_rank = int((flat > center_value).sum().item()) + 1
    return pd.DataFrame(rows), center_value, center_rank


fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
for row, name in enumerate(["conv_lr1", "conv_hr1"]):
    conv = getattr(model, name)
    logits = conv.spatial_logits.detach().cpu()
    prior = spatial_prior(conv)
    center_r = logits.shape[0] // 2
    center_c = logits.shape[1] // 2

    ax0 = axes[row, 0]
    ax1 = axes[row, 1]
    im0 = ax0.imshow(logits, cmap="coolwarm")
    im1 = ax1.imshow(prior, cmap="viridis")
    ax0.scatter([center_c], [center_r], c="k", marker="x", s=80)
    ax1.scatter([center_c], [center_r], c="r", marker="x", s=80)
    ax0.set_title(f"{name} spatial_logits")
    ax1.set_title(f"{name} softmax(spatial_logits)")
    for ax in (ax0, ax1):
        ax.set_xlabel("col")
        ax.set_ylabel("row")
    fig.colorbar(im0, ax=ax0, fraction=0.046)
    fig.colorbar(im1, ax=ax1, fraction=0.046)

plt.show()

for name in ["conv_lr1", "conv_hr1"]:
    conv = getattr(model, name)
    top_df, center_value, center_rank = top_kernel_rows(conv)
    print(f"\n{name} top spatial-prior entries")
    display(top_df)
    print(f"center prior weight: {center_value:.6f} (rank {center_rank})")


conv_lr1 top spatial-prior entries


/tmp/ipykernel_1177588/3387157958.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,rank,row,col,offset,prior_weight
0,1,4,4,"(2, 2)",0.088947
1,2,3,4,"(1, 2)",0.068142
2,3,4,3,"(2, 1)",0.067743
3,4,4,2,"(2, 0)",0.051839
4,5,4,1,"(2, -1)",0.047778
5,6,1,4,"(-1, 2)",0.046702


center prior weight: 0.009571 (rank 25)

conv_hr1 top spatial-prior entries


,rank,row,col,offset,prior_weight
0,1,3,0,"(0, -3)",0.052254
1,2,3,1,"(0, -2)",0.051089
2,3,3,6,"(0, 3)",0.041402
3,4,1,3,"(-2, 0)",0.036977
4,5,0,3,"(-3, 0)",0.031611
5,6,2,6,"(-1, 3)",0.031540


center prior weight: 0.007313 (rank 49)


## Decode the TP weights by coupling path

Each `tp.weight[i]` corresponds to one allowed equivariant coupling instruction.

Here we map every learned TP weight to:

- input irrep block from the center feature
- input irrep block from the neighborhood summary
- output irrep block
- built-in `path_weight` normalization used by `e3nn`
- learned weight from the checkpoint
- `effective_scale = path_weight * learned_weight` as a convenient combined number

In [6]:
def tp_instruction_table(conv):
    in1_blocks = [str(mul_ir) for mul_ir in conv.tp.irreps_in1]
    in2_blocks = [str(mul_ir) for mul_ir in conv.tp.irreps_in2]
    out_blocks = [str(mul_ir) for mul_ir in conv.tp.irreps_out]
    learned = conv.tp.weight.detach().cpu().reshape(-1)

    rows = []
    for idx, inst in enumerate(conv.tp.instructions):
        weight = float(learned[idx].item())
        rows.append(
            {
                "path_idx": idx,
                "in1": in1_blocks[inst.i_in1],
                "in2": in2_blocks[inst.i_in2],
                "out": out_blocks[inst.i_out],
                "connection_mode": inst.connection_mode,
                "path_weight": float(inst.path_weight),
                "learned_weight": weight,
                "effective_scale": float(inst.path_weight * weight),
            }
        )
    return pd.DataFrame(rows)


lr_tp_df = tp_instruction_table(model.conv_lr1)
hr_tp_df = tp_instruction_table(model.conv_hr1)

print("conv_lr1 TP paths")
display(lr_tp_df)
print("conv_hr1 TP paths")
display(hr_tp_df)

conv_lr1 TP paths


,path_idx,in1,in2,out,connection_mode,path_weight,learned_weight,effective_scale
0,0,1x2e,1x2e,1x2e,uvw,1.118034,0.346404,0.387292
1,1,1x2e,1x2e,1x4e,uvw,1.500000,0.635382,0.953073
2,2,1x2e,1x4e,1x2e,uvw,1.118034,0.257707,0.288125
3,3,1x2e,1x4e,1x4e,uvw,1.500000,-0.814178,-1.221266
4,4,1x4e,1x2e,1x2e,uvw,1.118034,-1.437236,-1.606879
5,5,1x4e,1x2e,1x4e,uvw,1.500000,-0.340153,-0.510230
6,6,1x4e,1x4e,1x2e,uvw,1.118034,2.031757,2.271574
7,7,1x4e,1x4e,1x4e,uvw,1.500000,0.976806,1.465209


conv_hr1 TP paths


,path_idx,in1,in2,out,connection_mode,path_weight,learned_weight,effective_scale
0,0,1x2e,1x2e,1x2e,uvw,1.118034,0.095232,0.106473
1,1,1x2e,1x2e,1x4e,uvw,1.500000,0.952840,1.429261
2,2,1x2e,1x4e,1x2e,uvw,1.118034,0.155890,0.174290
3,3,1x2e,1x4e,1x4e,uvw,1.500000,0.545209,0.817814
4,4,1x4e,1x2e,1x2e,uvw,1.118034,0.934042,1.044291
5,5,1x4e,1x2e,1x4e,uvw,1.500000,-0.924860,-1.387289
6,6,1x4e,1x4e,1x2e,uvw,1.118034,1.335708,1.493367
7,7,1x4e,1x4e,1x4e,uvw,1.500000,-0.861074,-1.291612


In [7]:
tp_compare = lr_tp_df[["path_idx", "in1", "in2", "out", "learned_weight", "effective_scale"]].merge(
    hr_tp_df[["path_idx", "learned_weight", "effective_scale"]],
    on="path_idx",
    suffixes=("_lr", "_hr"),
)
display(tp_compare)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = list(range(len(tp_compare)))

axes[0].bar([i - 0.18 for i in x], tp_compare["learned_weight_lr"], width=0.36, label="LR")
axes[0].bar([i + 0.18 for i in x], tp_compare["learned_weight_hr"], width=0.36, label="HR")
axes[0].axhline(0.0, color="black", linewidth=1)
axes[0].set_title("Raw TP weights")
axes[0].set_xlabel("path_idx")
axes[0].set_ylabel("learned_weight")
axes[0].set_xticks(x)
axes[0].legend()

axes[1].bar([i - 0.18 for i in x], tp_compare["effective_scale_lr"], width=0.36, label="LR")
axes[1].bar([i + 0.18 for i in x], tp_compare["effective_scale_hr"], width=0.36, label="HR")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_title("path_weight * learned_weight")
axes[1].set_xlabel("path_idx")
axes[1].set_ylabel("effective_scale")
axes[1].set_xticks(x)
axes[1].legend()

plt.show()

,path_idx,in1,in2,out,learned_weight_lr,effective_scale_lr,learned_weight_hr,effective_scale_hr
0,0,1x2e,1x2e,1x2e,0.346404,0.387292,0.095232,0.106473
1,1,1x2e,1x2e,1x4e,0.635382,0.953073,0.952840,1.429261
2,2,1x2e,1x4e,1x2e,0.257707,0.288125,0.155890,0.174290
3,3,1x2e,1x4e,1x4e,-0.814178,-1.221266,0.545209,0.817814
4,4,1x4e,1x2e,1x2e,-1.437236,-1.606879,0.934042,1.044291
5,5,1x4e,1x2e,1x4e,-0.340153,-0.510230,-0.924860,-1.387289
6,6,1x4e,1x4e,1x2e,2.031757,2.271574,1.335708,1.493367
7,7,1x4e,1x4e,1x4e,0.976806,1.465209,-0.861074,-1.291612


/tmp/ipykernel_1177588/478191942.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Best-checkpoint exploration summary

This final section turns the checkpoint into a compact interpretation.

In [8]:
def summary_rows(name, conv):
    prior = spatial_prior(conv)
    center_r = prior.shape[0] // 2
    center_c = prior.shape[1] // 2
    center_value = float(prior[center_r, center_c].item())
    flat = prior.reshape(-1)
    center_rank = int((flat > center_value).sum().item()) + 1
    return {
        "conv": name,
        "kernel_size": int(conv.kernel_size),
        "center_logit": float(conv.spatial_logits[center_r, center_c].item()),
        "center_prior_weight": center_value,
        "center_prior_rank": center_rank,
        "max_prior_weight": float(prior.max().item()),
        "min_prior_weight": float(prior.min().item()),
        "tp_weight_l1": float(conv.tp.weight.detach().abs().sum().item()),
        "tp_weight_l2": float(conv.tp.weight.detach().pow(2).sum().sqrt().item()),
    }


summary_df = pd.DataFrame(
    [
        summary_rows("conv_lr1", model.conv_lr1),
        summary_rows("conv_hr1", model.conv_hr1),
    ]
)
display(summary_df)

print("Observations to check against the tables/heatmaps above:")
print("- Both convs learn only two parameter tensors: spatial_logits and tp.weight.")
print("- The equivariant basis tensors stored under tp._compiled_main_left_right.* are fixed buffers, not learned weights.")
print("- LR and HR use the same 8 coupling paths, but they learn noticeably different TP coefficients.")
print("- The learned spatial priors are anisotropic rather than isotropic.")
print("- In this checkpoint, the center location is not the dominant spatial prior for either conv.")

,conv,kernel_size,center_logit,center_prior_weight,center_prior_rank,max_prior_weight,min_prior_weight,tp_weight_l1,tp_weight_l2
0,conv_lr1,5,-1.315707,0.009571,25,0.088947,0.009571,6.839624,2.918312
1,conv_hr1,7,-0.933927,0.007313,49,0.052254,0.007313,5.804855,2.343479


Observations to check against the tables/heatmaps above:
- Both convs learn only two parameter tensors: spatial_logits and tp.weight.
- The equivariant basis tensors stored under tp._compiled_main_left_right.* are fixed buffers, not learned weights.
- LR and HR use the same 8 coupling paths, but they learn noticeably different TP coefficients.
- The learned spatial priors are anisotropic rather than isotropic.
- In this checkpoint, the center location is not the dominant spatial prior for either conv.
